In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
CV = 5

ARTIFACTS_DIR = "artifacts"
FIGURES_DIR = os.path.join(ARTIFACTS_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

def compute_metrics(y_true, y_pred, y_proba=None):
    metrics = {}
    metrics["accuracy"] = float(accuracy_score(y_true, y_pred))
    try:
        if len(np.unique(y_true)) == 2:
            metrics["f1"] = float(f1_score(y_true, y_pred))
        else:
            metrics["f1_macro"] = float(f1_score(y_true, y_pred, average="macro"))
    except Exception:
        pass
    if y_proba is not None:
        try:
            if len(np.unique(y_true)) == 2:
                metrics["roc_auc"] = float(roc_auc_score(y_true, y_proba))
            else:
                metrics["roc_auc_ovr"] = float(roc_auc_score(y_true, y_proba, multi_class="ovr"))
        except Exception:
            pass
    return metrics

def save_figure(fig, fname):
    path = os.path.join(FIGURES_DIR, fname)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved figure:", path)

def save_json(obj, fname):
    path = os.path.join(ARTIFACTS_DIR, fname)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    print("Saved json:", path)


In [3]:
csv_path = "S06-hw-dataset-02.csv" 
df = pd.read_csv(csv_path)

display(df.head())
display(df.info())
display(df.describe(include="all"))

print("Target distribution (normalized):")
display(df["target"].value_counts(normalize=True))

,id,f01,f02,f03,f04,f05,f06,f07,f08,f09,...,f29,f30,f31,f32,f33,f34,f35,x_int_1,x_int_2,target
0,1,-0.149235,-2.826966,-0.522901,-4.198449,1.364943,0.815043,-1.195518,-1.932232,2.396353,...,-0.159323,0.448015,0.572745,0.149916,0.878392,-0.679733,1.412751,0.421883,9.217167,1
1,2,-1.966180,-4.877542,0.268367,-9.607791,0.097149,1.347185,-3.872575,-0.395117,1.710068,...,-0.389212,1.383794,0.169876,0.043969,-0.963545,1.006643,-2.488690,9.590124,24.772826,0
2,3,-0.555964,-0.999920,0.209673,-14.119498,-1.808950,-0.006222,-4.651108,0.911944,-0.289037,...,-1.383970,3.044321,-0.182864,1.425649,-8.418598,-4.629754,-0.439798,0.555919,41.800517,0
3,4,-2.049199,-5.600713,-1.664677,-6.263893,-5.224455,0.848351,1.407210,-0.542080,0.119102,...,-2.713080,2.762637,-0.520796,-0.142455,1.668338,2.292810,-10.744916,11.476977,65.315860,0
4,5,-0.220556,4.889479,-2.235840,6.450046,0.774389,-2.382625,2.584816,4.211856,-0.317889,...,-1.302872,2.478862,1.528610,1.098131,3.547087,2.517757,-9.364106,-1.078404,93.017870,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18000 entries, 0 to 17999
Data columns (total 39 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   id       18000 non-null  int64  
 1   f01      18000 non-null  float64
 2   f02      18000 non-null  float64
 3   f03      18000 non-null  float64
 4   f04      18000 non-null  float64
 5   f05      18000 non-null  float64
 6   f06      18000 non-null  float64
 7   f07      18000 non-null  float64
 8   f08      18000 non-null  float64
 9   f09      18000 non-null  float64
 10  f10      18000 non-null  float64
 11  f11      18000 non-null  float64
 12  f12      18000 non-null  float64
 13  f13      18000 non-null  float64
 14  f14      18000 non-null  float64
 15  f15      18000 non-null  float64
 16  f16      18000 non-null  float64
 17  f17      18000 non-null  float64
 18  f18      18000 non-null  float64
 19  f19      18000 non-null  float64
 20  f20      18000 non-null  float64
 21  f21      180

None

,id,f01,f02,f03,f04,f05,f06,f07,f08,f09,...,f29,f30,f31,f32,f33,f34,f35,x_int_1,x_int_2,target
count,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,...,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,18000.000000,1.800000e+04,18000.000000
mean,9000.500000,-0.418555,0.614251,0.004559,0.059000,0.405086,0.012123,-0.283473,-0.266880,0.255107,...,-0.139825,0.108568,0.007238,0.000904,-0.716862,-0.274520,0.344991,1.517339,2.576221e+01,0.262611
std,5196.296758,2.178005,3.926778,1.000134,5.713672,2.497581,0.987226,2.193891,2.081431,2.225776,...,2.148834,2.234315,0.997861,1.002115,3.913704,2.482890,4.927315,10.630850,5.423748e+01,0.440065
min,1.000000,-10.014698,-15.510323,-4.031762,-23.663256,-12.289308,-3.741536,-9.591425,-8.293319,-13.655742,...,-8.171469,-9.214171,-3.937091,-3.963063,-19.389908,-10.031559,-20.768452,-107.788145,1.895059e-07,0.000000
25%,4500.750000,-1.866134,-2.048192,-0.673127,-3.544964,-1.153000,-0.653090,-1.743214,-1.688121,-1.177480,...,-1.589638,-1.369266,-0.663023,-0.684164,-3.286842,-1.897893,-2.752685,-2.018750,1.226029e+00,0.000000
50%,9000.500000,-0.465100,0.600291,0.003581,0.072826,0.485625,0.018765,-0.251263,-0.302463,0.350739,...,-0.204785,0.158715,0.001912,-0.003157,-0.618472,-0.339901,0.573153,0.318011,6.581865e+00,0.000000
75%,13500.250000,0.966393,3.229850,0.671390,3.689490,2.075739,0.689304,1.195481,1.109589,1.764113,...,1.254595,1.600671,0.677296,0.676558,1.948803,1.314163,3.649794,4.212111,2.576847e+01,1.000000
max,18000.000000,9.589975,15.417329,3.817025,26.815691,10.665184,3.528280,7.794627,8.892834,8.699629,...,9.290667,8.794320,4.341030,3.781380,14.065595,10.639974,20.226291,94.891804,1.103449e+03,1.000000


Target distribution (normalized):


target
0    0.737389
1    0.262611
Name: proportion, dtype: float64

In [4]:
drop_cols = []
if "id" in df.columns:
    drop_cols.append("id")
X = df.drop(columns=drop_cols + ["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("X_train.shape:", X_train.shape, "X_test.shape:", X_test.shape)
print("Train target distribution:")
display(y_train.value_counts(normalize=True))
print("Test target distribution:")
display(y_test.value_counts(normalize=True))

X_train.shape: (14400, 37) X_test.shape: (3600, 37)
Train target distribution:


target
0    0.737361
1    0.262639
Name: proportion, dtype: float64

Test target distribution:


target
0    0.7375
1    0.2625
Name: proportion, dtype: float64

Фиксируем random_state для воспроизводимости; stratify сохраняет баланс классов

In [5]:
results = {}
search_summaries = {}

dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
y_dummy_pred = dummy.predict(X_test)

y_dummy_proba = dummy.predict_proba(X_test)[:, 1] if hasattr(dummy, "predict_proba") else None
results["DummyClassifier"] = compute_metrics(y_test, y_dummy_pred, y_dummy_proba)

pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

param_grid_lr = {"lr__C": [0.01, 0.1, 1.0, 10.0]}
grid_lr = GridSearchCV(pipe_lr, param_grid_lr, cv=CV, scoring="roc_auc" if len(np.unique(y))==2 else "f1_macro", n_jobs=-1)
grid_lr.fit(X_train, y_train)
best_lr = grid_lr.best_estimator_
y_lr_pred = best_lr.predict(X_test)
y_lr_proba = best_lr.predict_proba(X_test)[:, 1] if hasattr(best_lr, "predict_proba") else None
results["LogisticRegression"] = compute_metrics(y_test, y_lr_pred, y_lr_proba)
search_summaries["LogisticRegression"] = {
    "best_params": grid_lr.best_params_,
    "best_score": float(grid_lr.best_score_)
}

pd.DataFrame(results).T

,accuracy,f1,roc_auc
DummyClassifier,0.737500,0.000000,0.500000
LogisticRegression,0.811944,0.560675,0.797692


In [ ]:
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)

param_grid_dt = {
    "max_depth": [None, 5, 10, 15],
    "min_samples_leaf": [1, 5, 10],
}

grid_dt = GridSearchCV(dt, param_grid_dt, cv=CV, scoring="roc_auc" if len(np.unique(y))==2 else "f1_macro", n_jobs=-1)
grid_dt.fit(X_train, y_train)

best_dt = grid_dt.best_estimator_
y_dt_pred = best_dt.predict(X_test)
y_dt_proba = best_dt.predict_proba(X_test)[:, 1] if hasattr(best_dt, "predict_proba") else None

results["DecisionTree"] = compute_metrics(y_test, y_dt_pred, y_dt_proba)
search_summaries["DecisionTree"] = {
    "best_params": grid_dt.best_params_,
    "best_score": float(grid_dt.best_score_)
}

print("DecisionTree best params:", grid_dt.best_params_)
pd.DataFrame(results).T

DecisionTree best params: {'max_depth': 10, 'min_samples_leaf': 10}


,accuracy,f1,roc_auc
DummyClassifier,0.737500,0.000000,0.500000
LogisticRegression,0.811944,0.560675,0.797692
DecisionTree,0.838333,0.657647,0.837103


In [7]:
rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

param_grid_rf = {
    "n_estimators": [100, 300],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 3, 5],
    "max_features": ["sqrt", 0.5]
}

grid_rf = GridSearchCV(rf, param_grid_rf, cv=CV, scoring="roc_auc" if len(np.unique(y))==2 else "f1_macro", n_jobs=-1)
grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_
y_rf_pred = best_rf.predict(X_test)
y_rf_proba = best_rf.predict_proba(X_test)[:, 1] if hasattr(best_rf, "predict_proba") else None

results["RandomForest"] = compute_metrics(y_test, y_rf_pred, y_rf_proba)
search_summaries["RandomForest"] = {
    "best_params": grid_rf.best_params_,
    "best_score": float(grid_rf.best_score_)
}

print("RandomForest best params:", grid_rf.best_params_)
pd.DataFrame(results).T

RandomForest best params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 300}


,accuracy,f1,roc_auc
DummyClassifier,0.737500,0.000000,0.500000
LogisticRegression,0.811944,0.560675,0.797692
DecisionTree,0.838333,0.657647,0.837103
RandomForest,0.891944,0.760025,0.928686


In [8]:
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)

param_grid_gb = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5]
}

grid_gb = GridSearchCV(gb, param_grid_gb, cv=CV, scoring="roc_auc" if len(np.unique(y))==2 else "f1_macro", n_jobs=-1)
grid_gb.fit(X_train, y_train)

best_gb = grid_gb.best_estimator_
y_gb_pred = best_gb.predict(X_test)
y_gb_proba = best_gb.predict_proba(X_test)[:, 1] if hasattr(best_gb, "predict_proba") else None

results["GradientBoosting"] = compute_metrics(y_test, y_gb_pred, y_gb_proba)
search_summaries["GradientBoosting"] = {
    "best_params": grid_gb.best_params_,
    "best_score": float(grid_gb.best_score_)
}

print("GradientBoosting best params:", grid_gb.best_params_)
pd.DataFrame(results).T

GradientBoosting best params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}


,accuracy,f1,roc_auc
DummyClassifier,0.737500,0.000000,0.500000
LogisticRegression,0.811944,0.560675,0.797692
DecisionTree,0.838333,0.657647,0.837103
RandomForest,0.891944,0.760025,0.928686
GradientBoosting,0.900556,0.794725,0.927085


In [9]:
results_df = pd.DataFrame(results).T
display(results_df)

if "roc_auc" in results_df.columns:
    sort_col = "roc_auc"
elif "roc_auc_ovr" in results_df.columns:
    sort_col = "roc_auc_ovr"
elif "f1_macro" in results_df.columns:
    sort_col = "f1_macro"
else:
    sort_col = "f1" if "f1" in results_df.columns else "accuracy"

best_model_name = results_df[sort_col].idxmax()
print("Best model by", sort_col, ":", best_model_name)

metrics_test = results
save_json(metrics_test, "metrics_test.json")

,accuracy,f1,roc_auc
DummyClassifier,0.737500,0.000000,0.500000
LogisticRegression,0.811944,0.560675,0.797692
DecisionTree,0.838333,0.657647,0.837103
RandomForest,0.891944,0.760025,0.928686
GradientBoosting,0.900556,0.794725,0.927085


Best model by roc_auc : RandomForest
Saved json: artifacts\metrics_test.json


In [10]:
save_json(search_summaries, "search_summaries.json")

Saved json: artifacts\search_summaries.json


In [11]:
model_map = {
    "DummyClassifier": dummy,
    "LogisticRegression": best_lr,
    "DecisionTree": best_dt,
    "RandomForest": best_rf,
    "GradientBoosting": best_gb,
    "Stacking": stack if 'stack' in globals() else None
}

best_model = model_map.get(best_model_name)

joblib.dump(best_model, os.path.join(ARTIFACTS_DIR, "best_model.joblib"))
print("Saved best_model.joblib")

best_model_meta = {
    "best_model_name": best_model_name,
    "best_model_params": search_summaries.get(best_model_name, {}).get("best_params", None),
    "metric_used": sort_col,
    "metrics_on_test": results[best_model_name]
}
save_json(best_model_meta, "best_model_meta.json")

Saved best_model.joblib
Saved json: artifacts\best_model_meta.json


In [12]:
from sklearn.metrics import roc_curve, auc, ConfusionMatrixDisplay

y_best_pred = best_model.predict(X_test)
y_best_proba = best_model.predict_proba(X_test)[:, 1] if hasattr(best_model, "predict_proba") else None

cm = confusion_matrix(y_test, y_best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
fig1, ax1 = plt.subplots(figsize=(5,4))
disp.plot(ax=ax1)
ax1.set_title(f"Confusion Matrix: {best_model_name}")
save_figure(fig1, "confusion_matrix.png")
plt.close(fig1)

if y_best_proba is not None and len(np.unique(y_test)) == 2:
    fpr, tpr, _ = roc_curve(y_test, y_best_proba)
    roc_auc_val = auc(fpr, tpr)
    fig2, ax2 = plt.subplots(figsize=(5,4))
    ax2.plot(fpr, tpr, label=f"AUC = {roc_auc_val:.3f}")
    ax2.plot([0,1], [0,1], linestyle="--", color="gray")
    ax2.set_xlabel("FPR")
    ax2.set_ylabel("TPR")
    ax2.set_title(f"ROC Curve: {best_model_name}")
    ax2.legend()
    save_figure(fig2, "roc_curve_best_model.png")
    plt.close(fig2)

try:
    perm = permutation_importance(best_model, X_test, y_test, n_repeats=8, random_state=RANDOM_STATE, scoring="roc_auc" if len(np.unique(y))==2 else "f1_macro")
    imp_mean = perm.importances_mean
    idx = np.argsort(imp_mean)[::-1][:15]
    fig3, ax3 = plt.subplots(figsize=(8,4))
    ax3.bar(range(len(idx)), imp_mean[idx])
    ax3.set_xticks(range(len(idx)))
    ax3.set_xticklabels([X.columns[i] for i in idx], rotation=60, ha="right")
    ax3.set_ylabel("mean importance (drop in metric)")
    ax3.set_title(f"Permutation importance: {best_model_name} (top-15)")
    save_figure(fig3, "permutation_importance.png")
    plt.close(fig3)
except Exception as e:
    print("Permutation importance failed:", e)

Saved figure: artifacts\figures\confusion_matrix.png
Saved figure: artifacts\figures\roc_curve_best_model.png
Saved figure: artifacts\figures\permutation_importance.png


In [13]:
comp_path = os.path.join(ARTIFACTS_DIR, "model_comparison.csv")
pd.DataFrame(results).T.to_csv(comp_path)
print("Saved model comparison to", comp_path)

print("Artifacts:", os.listdir(ARTIFACTS_DIR))
print("Figures:", os.listdir(FIGURES_DIR))

Saved model comparison to artifacts\model_comparison.csv
Artifacts: ['best_model.joblib', 'best_model_meta.json', 'figures', 'metrics_test.json', 'model_comparison.csv', 'search_summaries.json']
Figures: ['confusion_matrix.png', 'permutation_importance.png', 'roc_curve_best_model.png']
